In [ ]:
%pip install -q pinecone langchain langchain-aws langchain-text-splitters sentence-transformers boto3 python-dotenv

# Week 5 Thursday -- End-to-End RAG Pipelines with Pinecone

This session brings together every building block from the week into a **single, runnable RAG system**:

| Building Block | Where We Covered It | How It's Used Today |
|---|---|---|
| Text splitting | Wednesday (LangChain splitters) | Chunking documents before embedding |
| Pinecone | Tuesday (vector DB operations) | Storing & retrieving embeddings |
| LangChain + Bedrock | Last week | Generating answers from retrieved context |

### Learning Objectives

By the end of this notebook you will be able to:

1. Preprocess raw documents (fix encoding, strip HTML, normalize whitespace) so embeddings capture *meaning* instead of *noise*.
2. Compare chunking strategies (fixed-size, sentence-based, recursive) and understand their trade-offs.
3. Build a **complete RAG pipeline**: ingest, embed, store in Pinecone, retrieve, and generate with Amazon Bedrock.
4. Evaluate retrieval quality and understand how each stage affects the final answer.

---

| Stage | Focus | Key Idea |
|---|---|---|
| **Stage 1** | Document Preprocessing | Encoding fixes, HTML cleanup, whitespace normalization |
| **Stage 2** | Chunking Strategies Comparison | Fixed-size vs sentence-based vs recursive |
| **Stage 3** | Complete RAG Pipeline | Ingest, Embed, Store in Pinecone, Retrieve, Generate with Bedrock |

---

## Stage 1 - Document Preprocessing

> *"Garbage in, garbage out"* -- the most important rule in any data pipeline.

Embedding models are trained on clean, well-formatted text. When you hand them raw HTML, smart quotes, or double-spaced walls of whitespace, the model wastes embedding dimensions encoding **noise** rather than **meaning**.

The preprocessing pipeline runs in a fixed order:

```
Raw Text --> Encoding Normalization --> HTML/Markup Removal --> Whitespace Normalization --> Clean Text
```

We will build each step individually, then wire them into a reusable `TextPreprocessor` class.

In [ ]:
import re
import unicodedata
from html import unescape

# == PART 1 == Encoding Normalization ================================

print("=" * 70)
print("PART 1: Encoding Normalization")
print("=" * 70)

print()
print("Common encoding gremlins:")
print('  Smart quotes   --> Regular quotes')
print('  Em dashes      --> Regular dashes')
print('  Ellipsis char  --> Three dots')
print('  Non-breaking spaces --> Regular spaces')
print()


def normalize_encoding(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    replacements = {
        "\u2018": "'",  "\u2019": "'",
        "\u201c": '"',  "\u201d": '"',
        "\u2013": "-",  "\u2014": "-",
        "\u2026": "...", "\u00a0": " ",
        "\u200b": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


sample = "He said \u201cHello\u201d \u2014 that\u2019s nice\u2026 isn\u2019t it?"
print(f"Before: {repr(sample)}")
print(f"After : {repr(normalize_encoding(sample))}")
print()
print("Smart quotes and em-dashes are now standard ASCII.")
print("This helps embedding models process text consistently.")

In [ ]:
# == PART 2 == HTML Removal ==========================================

print("=" * 70)
print("PART 2: HTML Tag Removal")
print("=" * 70)

print()
print("HTML cleaning strategy:")
print("  1. Remove <script> and <style> blocks entirely")
print("  2. Replace block elements (<p>, <div>, <br>) with newlines")
print("  3. Strip all remaining tags")
print("  4. Decode HTML entities (&amp; -> &, &lt; -> <)")
print()

messy_html = (
    '<html>\n'
    '<head><script>alert("bad!");</script></head>\n'
    '<body>\n'
    '<h1>Machine Learning Overview</h1>\n'
    '<p>Machine learning is a subset of <b>artificial intelligence</b> that\n'
    'enables computers to learn from data.</p>\n'
    '<p>It is becoming increasingly important in today\'s connected world,\n'
    'with applications ranging from   recommendation systems   to autonomous vehicles.</p>\n'
    '</body>\n'
    '</html>'
)


def remove_html(text: str) -> str:
    text = unescape(text)
    text = re.sub(r'<script[^>]*>.*?</script>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<style[^>]*>.*?</style>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<(p|div|br|h[1-6]|li|tr|article|section)[^>]*>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', '', text)
    return text


cleaned = remove_html(messy_html)
print("After HTML removal:")
print("-" * 50)
print(cleaned)
print("-" * 50)
print()
print("Script tags: gone.  Paragraph tags became newlines.  Bold tags stripped, text kept.")

In [ ]:
# == PART 3 == Whitespace Normalization ==============================

print("=" * 70)
print("PART 3: Whitespace Normalization")
print("=" * 70)


def normalize_whitespace(text: str) -> str:
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    lines = [line.strip() for line in text.split('\n')]
    return '\n'.join(lines).strip()


result = normalize_whitespace(cleaned)
print()
print("After whitespace normalization:")
print("-" * 50)
print(result)
print("-" * 50)
print()
print("Multiple spaces collapsed.  Multiple newlines collapsed.  Leading/trailing whitespace gone.")

In [ ]:
# == PART 4 == Reusable Preprocessing Pipeline =======================

print("=" * 70)
print("PART 4: Complete TextPreprocessor Pipeline")
print("=" * 70)


class TextPreprocessor:
    def __init__(self, strip_html=True, strip_markdown=True, lowercase=False):
        self.strip_html = strip_html
        self.strip_markdown = strip_markdown
        self.lowercase = lowercase

    def process(self, text: str) -> str:
        text = self._normalize_encoding(text)
        if self.strip_html:
            text = self._remove_html(text)
        if self.strip_markdown:
            text = self._remove_markdown(text)
        text = self._normalize_whitespace(text)
        if self.lowercase:
            text = text.lower()
        return text

    def _normalize_encoding(self, text):
        text = unicodedata.normalize('NFC', text)
        for old, new in {
            '\u2018': "'", '\u2019': "'",
            '\u201c': '"', '\u201d': '"',
            '\u2013': '-', '\u2014': '-',
            '\u2026': '...', '\u00a0': ' ',
        }.items():
            text = text.replace(old, new)
        return text

    def _remove_html(self, text):
        text = unescape(text)
        text = re.sub(r'<script[^>]*>.*?</script>', '', text, flags=re.DOTALL)
        text = re.sub(r'<style[^>]*>.*?</style>', '', text, flags=re.DOTALL)
        text = re.sub(r'<(p|div|br|h[1-6]|li)[^>]*>', '\n', text, flags=re.IGNORECASE)
        text = re.sub(r'<[^>]+>', '', text)
        return text

    def _remove_markdown(self, text):
        text = re.sub(r'^#+\s+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
        text = re.sub(r'\*([^*]+)\*', r'\1', text)
        text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
        text = re.sub(r'`([^`]+)`', r'\1', text)
        return text

    def _normalize_whitespace(self, text):
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n\s*\n', '\n\n', text)
        lines = [line.strip() for line in text.split('\n')]
        return '\n'.join(lines).strip()


preprocessor = TextPreprocessor(strip_html=True, strip_markdown=True)
final = preprocessor.process(messy_html)

print()
print("Fully preprocessed document:")
print("-" * 50)
print(final)
print("-" * 50)
print()
print("Pipeline order: encoding -> HTML -> Markdown -> whitespace -> (optional) case")
print("This class is reusable -- we will plug it into the RAG pipeline in Stage 3.")

---

## Stage 2 - Chunking Strategies Comparison

Embedding models cap out at a few hundred tokens. Documents are usually much longer, so we *must* split them. The question is **how**.

```
         TOO SMALL              JUST RIGHT               TOO LARGE
      +-----------+         +---------------+        +-----------------+
      | "Machine" |         | "Machine      |        | [entire chapter]|
      +-----------+         |  learning is  |        +-----------------+
       Fragment!            |  a subset..." |          Too much noise
       No context           +---------------+
                             Complete thought
```

| Strategy | Idea | Tradeoff |
|---|---|---|
| Fixed-size | Split every *N* characters | Simple but breaks mid-sentence |
| Sentence-based | Split at sentence boundaries | Complete thoughts, variable sizes |
| Recursive (LangChain default) | Try paragraph, line, sentence, word splits | Adapts to content structure |

We will run all three on the same document and compare the results.

In [ ]:
# == PART 1 == Fixed-Size Chunking ===================================

from typing import List

SAMPLE_DOC = (
    'Machine Learning Fundamentals\n\n'
    'Machine learning is a subset of artificial intelligence that enables '
    'computers to learn from data without being explicitly programmed. '
    'This technology has transformed numerous industries.\n\n'
    'Supervised Learning\n\n'
    'In supervised learning, the algorithm learns from labeled training data. '
    'The model makes predictions based on input features and compares them '
    'against known correct outputs. Common applications include email spam '
    'detection, image classification, and price prediction.\n\n'
    'Unsupervised Learning\n\n'
    'Unsupervised learning works with unlabeled data. The algorithm tries '
    'to find hidden patterns or structures in the data. Clustering and '
    'dimensionality reduction are key techniques in this category. '
    'Applications include customer segmentation and anomaly detection.\n\n'
    'Deep Learning\n\n'
    'Deep learning is a specialized subset of machine learning that uses '
    'neural networks with many layers. These deep neural networks can learn '
    'complex patterns from large amounts of data. They power modern AI '
    'applications like natural language processing and computer vision.\n\n'
    'The key advantage of deep learning is feature learning - the model '
    'automatically discovers the representations needed for classification '
    'or detection. This eliminates the need for manual feature engineering.'
)

print(f'Sample document: {len(SAMPLE_DOC)} chars, {len(SAMPLE_DOC.split())} words')
print()


def fixed_size_chunk(text: str, chunk_size: int = 200, overlap: int = 0) -> List[str]:
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


no_overlap = fixed_size_chunk(SAMPLE_DOC, 200, 0)
with_overlap = fixed_size_chunk(SAMPLE_DOC, 200, 50)

print(f'Fixed-size (no overlap):      {len(no_overlap)} chunks')
print(f'Fixed-size (50-char overlap): {len(with_overlap)} chunks')
print()

for i, c in enumerate(no_overlap[:3]):
    print(f'  Chunk {i+1} ({len(c)} chars):')
    print(f'    {c[:70]}...')

print()
print('Notice how chunks can break mid-sentence or even mid-word.')

In [ ]:
# == PART 2 == Sentence-Based Chunking ===============================


def sentence_chunk(text: str, max_chunk_size: int = 300) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current, length = [], [], 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if length + len(s) > max_chunk_size and current:
            chunks.append(' '.join(current))
            current, length = [], 0
        current.append(s)
        length += len(s) + 1
    if current:
        chunks.append(' '.join(current))
    return chunks


sent_chunks = sentence_chunk(SAMPLE_DOC, 300)
print(f'Sentence-based chunking: {len(sent_chunks)} chunks')
print()

for i, c in enumerate(sent_chunks[:3]):
    ends_ok = c.strip()[-1] in '.!?'
    print(f'  Chunk {i+1} ({len(c)} chars, ends with sentence: {ends_ok})')
    print(f'    {c[:80]}...')
    print()

print('Every chunk ends with a complete sentence -- much better for retrieval.')

In [ ]:
# == PART 3 == Recursive Chunking (LangChain) ========================

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' '],
)

recursive_chunks = splitter.split_text(SAMPLE_DOC)
print(f'Recursive chunking: {len(recursive_chunks)} chunks')
print()

for i, c in enumerate(recursive_chunks[:4]):
    preview = c[:80].replace('\n', ' ')
    print(f'  Chunk {i+1} ({len(c)} chars):')
    print(f'    {preview}...')
    print()

print('LangChain tries the coarsest separator first (paragraph break), then falls')
print('back to finer splits only when a chunk still exceeds the size limit.')

In [ ]:
# == PART 4 == Side-by-Side Comparison ===============================

from dataclasses import dataclass


@dataclass
class ChunkStats:
    name: str
    count: int
    avg: float
    shortest: int
    longest: int
    pct_complete: float


def analyze(chunks: List[str], name: str) -> ChunkStats:
    sizes = [len(c) for c in chunks]
    complete = sum(1 for c in chunks if c.strip() and c.strip()[-1] in '.!?')
    return ChunkStats(
        name=name,
        count=len(chunks),
        avg=sum(sizes) / len(sizes),
        shortest=min(sizes),
        longest=max(sizes),
        pct_complete=complete / len(chunks) * 100,
    )


strategies = {
    'Fixed (no overlap)': fixed_size_chunk(SAMPLE_DOC, 200, 0),
    'Fixed (50 overlap)': fixed_size_chunk(SAMPLE_DOC, 200, 50),
    'Sentence-based':     sentence_chunk(SAMPLE_DOC, 300),
    'Recursive (LC)':     recursive_chunks,
}

header = f"{'Strategy':<22} {'Chunks':>6} {'Avg':>7} {'Min':>5} {'Max':>5} {'Complete':>9}"
print(header)
print('-' * 62)
for name, chunks in strategies.items():
    s = analyze(chunks, name)
    print(f'{s.name:<22} {s.count:>6} {s.avg:>7.0f} {s.shortest:>5} {s.longest:>5} {s.pct_complete:>8.0f}%')

print()
print('Fixed chunking is fast but often produces incomplete sentences.')
print('Sentence-based always ends on a sentence boundary.')
print('Recursive (LangChain default) balances structure and size.')
print('There is no universally best strategy -- test on YOUR data.')

---

## Stage 3 - Complete RAG Pipeline with Pinecone & Bedrock

Time to wire everything together into a production-style RAG system:

```
Documents --> Preprocess --> Chunk --> Embed --> Upsert to Pinecone
                                                       |
User Question --> Embed --> Query Pinecone --> Top-K Chunks --> Bedrock LLM --> Answer
```

**Components we will use:**

| Component | Tool |
|---|---|
| Preprocessing | `TextPreprocessor` from Stage 1 |
| Chunking | `RecursiveCharacterTextSplitter` from Stage 2 |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` (384-dim) |
| Vector store | **Pinecone** (serverless index) |
| LLM generation | **Amazon Bedrock** via `init_chat_model` |

In [ ]:
# == PART 1 == Setup =================================================

import os
from dotenv import load_dotenv
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from langchain.chat_models import init_chat_model

load_dotenv()

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
embedder = SentenceTransformer('all-MiniLM-L6-v2')
EMBED_DIM = embedder.get_sentence_embedding_dimension()
print(f'Embedding model loaded -- dimension: {EMBED_DIM}')

INDEX_NAME = 'week5-rag-demo'

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    from pinecone import ServerlessSpec
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1'),
    )
    print(f'Created Pinecone index: {INDEX_NAME}')
else:
    print(f'Pinecone index already exists: {INDEX_NAME}')

index = pc.Index(INDEX_NAME)
print(f'Index stats: {index.describe_index_stats()}')

In [ ]:
# == PART 2 == Knowledge Base Documents ==============================

DOCUMENTS = [
    {
        'id': 'ml_basics',
        'title': 'Machine Learning Fundamentals',
        'content': (
            'Machine learning is a subset of artificial intelligence that enables '
            'computers to learn from data without being explicitly programmed. The '
            'core idea is that systems can identify patterns in data and make '
            'decisions with minimal human intervention.\n\n'
            'There are three main types of machine learning: supervised learning, '
            'unsupervised learning, and reinforcement learning. Each approach has '
            'different use cases and requirements for training data.\n\n'
            'Supervised learning uses labeled data to train models. The algorithm '
            'learns the relationship between inputs and known outputs, then applies '
            'this knowledge to new, unseen data.'
        ),
    },
    {
        'id': 'neural_nets',
        'title': 'Neural Networks Explained',
        'content': (
            'Neural networks are computing systems inspired by biological neural '
            'networks in the human brain. They consist of interconnected nodes '
            '(neurons) organized in layers that process information.\n\n'
            'A typical neural network has an input layer, one or more hidden '
            'layers, and an output layer. Data flows forward through the network, '
            'with each layer transforming the information.\n\n'
            'Deep learning uses neural networks with many hidden layers (hence '
            'deep). These deep networks can learn complex patterns and '
            'representations that shallow networks cannot capture.\n\n'
            'Backpropagation is the algorithm used to train neural networks. It '
            'calculates gradients of the loss function with respect to each '
            'weight, allowing the network to improve through gradient descent.'
        ),
    },
    {
        'id': 'nlp_intro',
        'title': 'Natural Language Processing',
        'content': (
            'Natural Language Processing (NLP) is a field of AI focused on '
            'enabling computers to understand, interpret, and generate human '
            'language.\n\n'
            'Modern NLP relies heavily on deep learning, particularly transformer '
            'architectures. Models like BERT and GPT have revolutionized how '
            'machines process text.\n\n'
            'Key NLP tasks include text classification, named entity recognition, '
            'sentiment analysis, machine translation, and question answering.\n\n'
            'Embeddings are fundamental to NLP. They convert words or sentences '
            'into dense vector representations that capture semantic meaning. '
            'Similar concepts have similar embeddings.'
        ),
    },
    {
        'id': 'vector_dbs',
        'title': 'Vector Databases',
        'content': (
            'Vector databases are specialized systems designed to store and query '
            'high-dimensional vectors efficiently. They are essential for modern '
            'AI applications like semantic search and recommendation systems.\n\n'
            'Unlike traditional databases that use exact matching, vector databases '
            'use similarity search. They find vectors that are close to a query '
            'vector in high-dimensional space.\n\n'
            'Popular vector databases include Pinecone, Weaviate, Milvus, and '
            'Qdrant. Each has different strengths for various use cases and scale '
            'requirements.\n\n'
            'HNSW (Hierarchical Navigable Small World) is a common indexing '
            'algorithm used by vector databases. It enables fast approximate '
            'nearest neighbor search in high-dimensional spaces.'
        ),
    },
    {
        'id': 'rag_systems',
        'title': 'RAG Systems',
        'content': (
            'Retrieval-Augmented Generation (RAG) combines information retrieval '
            'with text generation. When a user asks a question, the system first '
            'retrieves relevant documents, then uses those documents as context '
            'for generation.\n\n'
            'RAG solves the knowledge limitation problem of language models. '
            'Instead of relying solely on training data, RAG systems can access '
            'external knowledge bases for up-to-date and domain-specific '
            'information.\n\n'
            'Building a RAG system requires several components: a document '
            'processor, chunking strategy, embedding model, vector database, '
            'retrieval logic, and a language model for generation.\n\n'
            'The quality of RAG answers depends heavily on retrieval quality. '
            'If the wrong documents are retrieved, the generated answer will be '
            'wrong too.'
        ),
    },
]

print(f'Knowledge base: {len(DOCUMENTS)} documents')
for doc in DOCUMENTS:
    word_count = len(doc['content'].split())
    print(f'  - {doc["title"]} ({word_count} words)')

In [ ]:
# == PART 3 == Ingestion Pipeline ====================================

print('=' * 70)
print('INGESTION PHASE')
print('  Documents -> Preprocess -> Chunk -> Embed -> Upsert to Pinecone')
print('=' * 70)

rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)
prep = TextPreprocessor(strip_html=False, strip_markdown=False)

vectors_to_upsert = []
chunk_counter = 0

for doc in DOCUMENTS:
    clean_text = prep.process(doc['content'])
    chunks = rag_splitter.split_text(clean_text)
    print(f'\n  {doc["title"]}:')
    print(f'    raw {len(doc["content"])} chars -> cleaned {len(clean_text)} chars -> {len(chunks)} chunks')

    embeddings = embedder.encode(chunks).tolist()

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        vectors_to_upsert.append({
            'id': f'{doc["id"]}_chunk_{i}',
            'values': emb,
            'metadata': {
                'source': doc['id'],
                'title': doc['title'],
                'chunk_index': i,
                'text': chunk,
            },
        })
        chunk_counter += 1

index.upsert(vectors=vectors_to_upsert)

print(f'\nUpserted {chunk_counter} vectors to Pinecone index: {INDEX_NAME}')
print(f'Index stats: {index.describe_index_stats()}')

In [ ]:
# == PART 4 == Retrieval Pipeline ====================================

print('=' * 70)
print('QUERY PHASE -- Retrieval from Pinecone')
print('=' * 70)


def retrieve(question: str, top_k: int = 3):
    q_embedding = embedder.encode([question]).tolist()[0]
    results = index.query(vector=q_embedding, top_k=top_k, include_metadata=True)

    retrieved = []
    for match in results.matches:
        retrieved.append({
            'score': match.score,
            'title': match.metadata['title'],
            'text':  match.metadata['text'],
        })
    return retrieved


question = 'What is machine learning?'
hits = retrieve(question)

print(f'\nQuestion: {question}\n')
for i, hit in enumerate(hits):
    print(f'  [{i+1}] {hit["title"]}  (score {hit["score"]:.4f})')
    print(f'      {hit["text"][:100]}...\n')

In [ ]:
# == PART 5 == Generation with Amazon Bedrock ========================

print('=' * 70)
print('RAG GENERATION -- Bedrock Nova Lite')
print('=' * 70)

llm = init_chat_model('bedrock:us.amazon.nova-2-lite-v1:0')


def build_context(hits):
    parts = []
    for hit in hits:
        parts.append(f'[Source: {hit["title"]}]\n{hit["text"]}')
    return '\n\n---\n\n'.join(parts)


def rag_answer(question: str, top_k: int = 3) -> str:
    hits = retrieve(question, top_k=top_k)
    context = build_context(hits)

    prompt = (
        'Based on the following context, answer the question. '
        'If the context does not contain enough information, say so.\n\n'
        f'CONTEXT:\n{context}\n\n'
        f'QUESTION: {question}\n\n'
        'ANSWER:'
    )

    response = llm.invoke(prompt)
    return response.content


answer = rag_answer('What is machine learning?')
print(f'\nQuestion: What is machine learning?\n')
print(f'Answer:\n{answer}')

In [ ]:
# == PART 6 == Test with Multiple Queries ============================

test_questions = [
    'How does backpropagation work?',
    'What are embeddings used for in NLP?',
    'What is the advantage of RAG over fine-tuning?',
    'What indexing algorithm do vector databases use?',
]

for q in test_questions:
    print('=' * 60)
    print(f'Q: {q}\n')

    hits = retrieve(q, top_k=2)
    for i, h in enumerate(hits):
        print(f'  Retrieved [{i+1}]: {h["title"]}  (score {h["score"]:.4f})')

    answer = rag_answer(q, top_k=2)
    print(f'\n  Answer: {answer}\n')

---

## Key Takeaways

1. **Preprocessing is foundational.** Encoding normalization, HTML stripping, and whitespace cleanup prevent the embedding model from wasting dimensions on noise.

2. **Chunking strategy matters.** Fixed-size is fast but crude; sentence-based preserves grammar; recursive (LangChain's default) balances structure and size. *There is no universally best strategy -- test on your data.*

3. **The RAG pipeline has two phases:**
   - **Ingestion (offline):** Documents --> Preprocess --> Chunk --> Embed --> Store in Pinecone
   - **Query (online):** Question --> Embed --> Search Pinecone --> Retrieve Top-K --> Generate with Bedrock

4. **Quality compounds across stages.** Bad preprocessing leads to bad embeddings leads to bad retrieval leads to bad answers. Each step amplifies the next.

5. **Pinecone as the bridge.** The vector database connects ingestion and query. Once vectors are stored, retrieval is a single API call -- no matter how many documents you have.

---

### Up Next -- Friday: Evaluation Harness Engineering

How do you *know* your RAG pipeline is good? Friday's session builds an evaluation harness that measures:

- **Retrieval quality:** Are the right chunks coming back?
- **Answer relevance:** Does the generated answer address the question?
- **Faithfulness:** Is the answer grounded in the retrieved context, or is the LLM hallucinating?

We will use systematic evaluation to move from *"it seems to work"* to *"we can prove it works."*